# 20. Baseline — Valentine (Cupid · COMA · DistributionBased)

schema matching 표준 벤치마크인 Valentine(Koutras et al., ICDE 2021)의 구현으로 세 가지 방법을 실행한다.
DB 안의 모든 테이블 쌍에 대해 매칭 점수를 구하고, CRIM 과 같은 후보쌍·지표로 평가한다. `db_name` 을 DB 마다 바꿔 실행한다.

| 모델 | 사용하는 정보 | 원 논문 |
|---|---|---|
| Cupid | 컬럼명의 언어적 유사도 + 스키마 구조 (값 미사용) | Madhavan et al., VLDB 2001 |
| COMA | 이름·타입·값 인스턴스를 조합한 matcher 앙상블 (`use_instances=True`) | Do & Rahm, VLDB 2002 |
| DistributionBased | 컬럼 값 분포의 EMD 거리 (이름 미사용) | Zhang et al., SIGMOD 2011 |

- COMA 는 Java 8+ 가 필요하다.
- 출력: `workspace/valentine_ranking_metrics_<db>[_mask].json`

## 0. 설치

In [ ]:
# # Valentine 설치 (COMA는 Java 필요)
# !pip install valentine -q

# # COMA는 Java 8+ 필요 — Colab에는 기본 탑재
# !java -version 2>&1 | head -1
# # !apt-get install -y default-jre -q

In [ ]:
import os
java_home = "/opt/homebrew/Cellar/openjdk/25.0.2/libexec/openjdk.jdk/Contents/Home"
os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = java_home + "/bin:" + os.environ["PATH"]


## 1. 데이터 로드

In [ ]:
import os
import json
import time
import itertools
import pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from valentine import valentine_match
from valentine.algorithms import Cupid, Coma, DistributionBased, JaccardDistanceMatcher
from valentine.algorithms.jaccard_distance import StringDistanceFunction

# ── 폴더 경로 설정 ──────────────────────────────────────────
db_name = "toxicology"  # toxicology / california_schools / superhero / thrombosis_prediction (+ _mask)

FOLDER_PATH = f"../workspace/dataset_mini/bird_data_tables/{db_name}"
GT_PATH = f"../workspace/dataset_mini/bird_data_tables/{db_name}/ground_truth.json"

# 폴더 내 모든 CSV 파일 재귀 탐색
csv_files = sorted(pathlib.Path(FOLDER_PATH).rglob('*.csv'))
csv_files = [f for f in csv_files if f.name != '.csv']  # 숨김 파일 제외

# 모든 (A, B) 조합 생성 — 순서 무관하므로 combinations 사용
csv_pairs = list(itertools.combinations(csv_files, 2))

print(f"[폴더 스캔 완료]")
print(f"  폴더: {FOLDER_PATH}")
print(f"  발견된 CSV 파일: {len(csv_files)}개")
print(f"  생성된 쌍: {len(csv_pairs)}개")
print(f"\n[CSV 파일 목록]")
for i, f in enumerate(csv_files, 1):
    print(f"  {i:2d}. {f.relative_to(FOLDER_PATH)}")


## 2. Ground Truth 로드

JSON 포맷 예시:
```json
[
  ["orders.o_orderkey", "lineitem.l_orderkey"],
  ["orders.o_custkey",  "lineitem.l_suppkey"]
]
```
또는 딕셔너리 형식:
```json
{"matches": [["o_orderkey", "l_orderkey"], ...]}
```

In [ ]:
# ── Ground Truth 로드 ──────────────────────────────────────
with open(GT_PATH, 'r') as f:
    gt_all = json.load(f)

print(f"\n[Ground Truth 로드 완료]")
print(f"  파일: {GT_PATH}")
print(f"  총 GT 쌍: {len(gt_all)}개")
print(f"\n[GT 쌍 목록]")
for pair in gt_all:
    print(f"  {pair[0]} ↔ {pair[1]}")

def get_pair_gt(gt_list, name_a, name_b):
    """
    전체 GT 목록에서 특정 (A, B) 쌍에 관련된 항목만 추출.
    GT 형식: ["table_a.col_a", "table_b.col_b"]
    name_a, name_b는 파일명(확장자 제외)
    반환: Valentine 형식 {(table_a, col_a): (table_b, col_b), ...}
    """
    filtered = {}
    for item in gt_list:
        ta, ca = item[0].split('.')
        tb, cb = item[1].split('.')
        
        # 파일명으로 테이블명 매핑 (파일명 = 테이블명)
        if set([ta, tb]) == set([name_a, name_b]):
            # (A의 컬럼, B의 컬럼) 순서로 정렬
            if ta == name_a:
                filtered[(ta, ca)] = (tb, cb)
            else:
                filtered[(tb, cb)] = (ta, ca)
    return filtered

print(f"\n[prepare_pair_gt() 함수 준비 완료]")


## 3. Valentine 매칭 실행

### 모델별 파라미터 설명

| 모델 | 주요 파라미터 | 기본값 |
|---|---|---|
| `Cupid` | `w_struct` (구조 가중치), `leaf_w_struct` (리프 구조 가중치), `th_accept` (수락 임계값) | 0.2, 0.6, 0.7 |
| `Coma` | `max_n` (최대 매칭 수), `use_instances` (인스턴스 사용 여부) | 0, True |
| `DistributionBased` | `threshold1` (1차 임계값), `threshold2` (2차 임계값) | 0.15, 0.15 |
| `JaccardDistanceMatcher` | `threshold_dist` (Levenshtein 임계값), `distance_fun` (거리 함수) | 0.8, Levenshtein |

In [ ]:
import traceback

def clean_df_for_valentine(df, max_rows=1000):
    """Valentine 입력용 DataFrame 전처리"""
    df = df.dropna(how='all', axis=0).dropna(how='all', axis=1)
    df.columns = [str(c).strip() for c in df.columns]
    if len(df) > max_rows:
        df = df.sample(max_rows, random_state=42)
    return df


def run_matcher(matcher, df_a, df_b, name_a, name_b, gt, model_name, pair_idx, total_pairs):
    """Valentine 매처 실행 후 결과 반환"""
    print(f"\n{'='*60}")
    print(f"  [{pair_idx}/{total_pairs}] {name_a} × {name_b}")
    print(f"  [{model_name}] 실행 중...")
    print(f"{'='*60}")

    # Valentine 내부에서 table_guid[0], table_guid[1] 로 2글자 접근함
    # 테이블 이름이 1글자면 IndexError → 최소 2글자로 패딩
    name_a_v = (name_a + "_") if len(name_a) < 2 else name_a
    name_b_v = (name_b + "_") if len(name_b) < 2 else name_b

    start = time.time()
    try:
        matches = valentine_match(df_a, df_b, matcher, name_a_v, name_b_v)
    except (ValueError, Exception) as e:
        elapsed = time.time() - start
        print(f"  [SKIP] {model_name} 실행 실패 ({elapsed:.2f}초): {type(e).__name__}: {e}")
        tb_lines = traceback.format_exc().splitlines()
        print(f"  [TRACEBACK] (최근 6줄)")
        for line in tb_lines[-6:]:
            print(f"    {line}")
        return None
    elapsed = time.time() - start

    # 패딩된 이름을 원래 이름으로 복원
    # matches.items() → ((ta, ca), (tb, cb)), score 형태
    restored = {}
    for key, score in matches.items():
        (ta, ca), (tb, cb) = key
        ta_orig = name_a if ta == name_a_v else (name_b if ta == name_b_v else ta)
        tb_orig = name_b if tb == name_b_v else (name_a if tb == name_a_v else tb)
        restored[((ta_orig, ca), (tb_orig, cb))] = score
    matches = type(matches)(restored)

    print(f"[후보 매칭 수]: {len(matches)}, [시간]: {elapsed:.2f}초")
    
    # Metrics 계산
    gt_list = [(col_a, col_b) for (_, col_a), (_, col_b) in gt.items()]
    print(gt_list)
    metrics = matches.get_metrics(gt_list) if gt_list else {}
    
    # ── F1 수정: Valentine 라이브러리 버그 대응 ──────────────────
    if metrics:
        p = metrics.get('Precision', 0.0)
        r = metrics.get('Recall', 0.0)
        f1_lib = metrics.get('F1 Score', 0.0)
        f1_manual = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
        if abs(f1_lib) < 1e-6 and f1_manual > 1e-6:
            metrics['F1 Score'] = f1_manual
            print(f"[F1 수정] Valentine F1=0 → 계산된 F1={f1_manual:.4f}")
        print(f"[Metrics] Precision={p:.4f}, Recall={r:.4f}, F1={metrics.get('F1 Score', 0):.4f}")
    else:
        print(f"[Metrics] GT가 없어 메트릭 계산 불가")

    sorted_matches = sorted(matches.items(), key=lambda x: x[1], reverse=True)
    
    # ── TP/FP/TN/FN 계산 ──────────────────────────────────────
    total_possible = len(df_a.columns) * len(df_b.columns)
    
    if len(gt) > 0:
        gt_set = {(col_a, col_b) for (_, col_a), (_, col_b) in gt.items()}
        tp = sum(1 for (_, col_a), (_, col_b) in matches.keys() if (col_a, col_b) in gt_set)
        fp = len(matches) - tp
        fn = len(gt) - tp
        tn = total_possible - tp - fp - fn
    else:
        tp = 0
        fp = len(matches)
        fn = 0
        tn = total_possible - fp
    
    return {
        'pair': f'{name_a} × {name_b}',
        'name_a': name_a,
        'name_b': name_b,
        'model': model_name,
        'matches': matches,
        'metrics': metrics,
        'time': elapsed,
        'sorted_matches': sorted_matches,
        'has_gt': len(gt) > 0,
        'num_gt': len(gt),
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn,
        'total_possible': total_possible
    }

all_results = []

## 3. Valentine 배치 실행

폴더 내 모든 CSV 쌍에 대해 4개 모델을 순차 실행합니다.


In [ ]:
# ── 4개 모델 정의 ─────────────────────────────────────────
matchers = {
    'cupid': Cupid(w_struct=0.2, leaf_w_struct=0.6, th_accept=0.7),
    'coma': Coma(max_n=0, use_instances=True, java_xmx="8192m"),
    'dist': DistributionBased(threshold1=0.15, threshold2=0.15),
    # 'jaccard': JaccardDistanceMatcher(threshold_dist=0.8, distance_fun=StringDistanceFunction.Levenshtein)
}

# ── 배치 실행 루프 ─────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  배치 실행 시작: {len(csv_pairs)}개 쌍 × {len(matchers)}개 모델 = {len(csv_pairs)*len(matchers)}개 작업")
print(f"{'='*70}")

for pair_idx, (path_a, path_b) in enumerate(csv_pairs, 1):
    name_a, name_b = path_a.stem, path_b.stem
    
    # CSV 로드 + 전처리
    try:
        df_a = clean_df_for_valentine(pd.read_csv(path_a))
        df_b = clean_df_for_valentine(pd.read_csv(path_b))
    except Exception as e:
        print(f"\n[오류] {name_a} or {name_b} 로드 실패: {e}")
        continue
    
    # 이 쌍의 GT 필터링
    pair_gt = get_pair_gt(gt_all, name_a, name_b)
    
    # 모델 순차 실행
    for model_key, matcher in matchers.items():
        result = run_matcher(
            matcher, df_a, df_b, name_a, name_b, pair_gt, 
            f"{model_key.upper()}", pair_idx, len(csv_pairs)
        )
        if result is not None:
            all_results.append(result)

print(f"\n{'='*70}")
print(f"  배치 실행 완료: {len(all_results)}개 결과 수집")
print(f"{'='*70}")

## 4. 결과 집계 및 요약

GT가 존재하는 쌍만 대상으로 모델별 평균 성능 지표를 계산합니다.


In [ ]:
# ── 4-1. 모델별 평균 성능 (GT 있는 쌍만) ────────────────
df_all = pd.DataFrame(all_results)

print(f"\n[전체 결과 요약]")
print(f"  총 작업 수: {len(all_results)}")
print(f"  GT 있는 쌍: {df_all['has_gt'].sum()}, GT 없는 쌍: {(~df_all['has_gt']).sum()}")

# GT 있는 쌍만 필터
df_gt_pairs = df_all[df_all['has_gt'] == True].copy()

if len(df_gt_pairs) > 0:
    # 메트릭 추출 (Precision, Recall, F1 Score)
    metrics_df_list = []
    for _, row in df_gt_pairs.iterrows():
        metrics = row['metrics']
        metrics_df_list.append({
            'pair': row['pair'],
            'model': row['model'],
            'Precision': metrics.get('Precision', 0.0),
            'Recall': metrics.get('Recall', 0.0),
            'F1 Score': metrics.get('F1 Score', 0.0),
            'time': row['time']
        })
    
    df_metrics = pd.DataFrame(metrics_df_list)
    
    # 모델별 평균
    model_summary = df_metrics.groupby('model')[['Precision', 'Recall', 'F1 Score', 'time']].agg({
        'Precision': ['mean', 'std'],
        'Recall': ['mean', 'std'],
        'F1 Score': ['mean', 'std'],
        'time': ['mean', 'sum']
    }).round(4)
    
    print(f"\n[모델별 평균 성능 (GT 있는 쌍만, {len(df_gt_pairs)}개 작업)]")
    print(model_summary)
    
    # 플랫 테이블 형식
    print(f"\n[간결 요약 테이블]")
    summary_flat = df_metrics.groupby('model')[['Precision', 'Recall', 'F1 Score']].mean().round(4)
    print(summary_flat)
else:
    print(f"\n[경고] GT가 있는 쌍이 없어 성능 지표를 계산할 수 없습니다.")

# ── 4-2. GT 없는 쌍 목록 ────────────────────────────────
df_no_gt = df_all[df_all['has_gt'] == False][['pair', 'model']].drop_duplicates()
if len(df_no_gt) > 0:
    print(f"\n[GT 없는 쌍 목록 ({len(df_no_gt)//4}개 쌍)]")
    for pair in df_no_gt['pair'].unique():
        print(f"  - {pair}")


In [ ]:
# ── 4-3. 시각화: 모델별 F1 점수 비교 ────────────────────
if len(df_metrics) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # (1) 모델별 평균 F1
    f1_by_model = df_metrics.groupby('model')['F1 Score'].mean().sort_values(ascending=False)
    colors = ['#2E75B6', '#ED7D31', '#70AD47', '#7030A0']
    f1_by_model.plot(kind='bar', ax=axes[0], color=colors[:len(f1_by_model)])
    axes[0].set_title('모델별 평균 F1 Score (GT 있는 쌍만)', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('F1 Score')
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)
    axes[0].grid(axis='y', alpha=0.3)
    
    # (2) 모델별 Precision, Recall, F1
    metrics_by_model = df_metrics.groupby('model')[['Precision', 'Recall', 'F1 Score']].mean()
    metrics_by_model.plot(kind='bar', ax=axes[1])
    axes[1].set_title('모델별 성능 지표', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Score')
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)
    axes[1].legend(loc='best')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()


In [ ]:
# ── 4-4. 쌍별 상세 결과 ────────────────────────────────────
print(f"\n[쌍별 상세 결과 (GT 있는 쌍만)]")
print(f"{'Pair':<30} {'Model':<8} {'Precision':<12} {'Recall':<12} {'F1':<12}")
print(f"{'-'*75}")

for pair_name in sorted(df_metrics['pair'].unique()):
    pair_data = df_metrics[df_metrics['pair'] == pair_name]
    for _, row in pair_data.iterrows():
        print(f"{row['pair']:<30} {row['model']:<8} "
              f"{row['Precision']:<12.4f} {row['Recall']:<12.4f} {row['F1 Score']:<12.4f}")


In [ ]:
# ── 4-5. 결과 JSON 저장 ────────────────────────────────────
output = {
    'summary': {
        'total_pairs': len(csv_pairs),
        'gt_pairs': len(df_gt_pairs) // 4 if len(df_metrics) > 0 else 0,
        'total_tasks': len(all_results),
        'models': list(matchers.keys())
    },
    'per_pair': []
}

# 쌍별 집계
for pair_name in df_metrics['pair'].unique():
    pair_data = df_metrics[df_metrics['pair'] == pair_name]
    pair_info = {
        'pair': pair_name,
        'results': []
    }
    for _, row in pair_data.iterrows():
        pair_info['results'].append({
            'model': row['model'],
            'Precision': float(row['Precision']),
            'Recall': float(row['Recall']),
            'F1 Score': float(row['F1 Score']),
            'time_seconds': float(row['time'])
        })
    output['per_pair'].append(pair_info)

# 모델별 평균
output['model_average'] = {}
for model in df_metrics['model'].unique():
    model_data = df_metrics[df_metrics['model'] == model]
    output['model_average'][model] = {
        'Precision_mean': float(model_data['Precision'].mean()),
        'Precision_std': float(model_data['Precision'].std()),
        'Recall_mean': float(model_data['Recall'].mean()),
        'Recall_std': float(model_data['Recall'].std()),
        'F1_Score_mean': float(model_data['F1 Score'].mean()),
        'F1_Score_std': float(model_data['F1 Score'].std()),
        'time_total': float(model_data['time'].sum()),
        'time_mean': float(model_data['time'].mean())
    }

output_path = '../workspace/valentine_results_batch.json'
with open(output_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f"\n[결과 저장 완료]")
print(f"  파일: {output_path}")
print(f"  크기: {len(json.dumps(output))} bytes")


In [ ]:
# ── 모델별 TP/FP/TN/FN 합계 ────────────────────────────────────
print(f"\n{'='*70}")
print(f"[모델별 TP/FP/TN/FN 합계 (전체 쌍 포함, GT 없는 쌍은 FP/TN으로 처리)]")
print(f"{'='*70}")

# 전체 결과 (GT 유무 무관)
confusion_by_model = {}
for result in all_results:
    model = result['model']
    if model not in confusion_by_model:
        confusion_by_model[model] = {'TP': 0, 'FP': 0, 'TN': 0, 'FN': 0, 'total': 0}
    
    confusion_by_model[model]['TP'] += result['tp']
    confusion_by_model[model]['FP'] += result['fp']
    confusion_by_model[model]['TN'] += result['tn']
    confusion_by_model[model]['FN'] += result['fn']
    confusion_by_model[model]['total'] += result['total_possible']

# 테이블 출력
print(f"\n{'Model':<10} {'TP':<8} {'FP':<8} {'TN':<12} {'FN':<8} {'합계':<10}")
print(f"{'-'*60}")
for model in sorted(confusion_by_model.keys()):
    counts = confusion_by_model[model]
    total = counts['TP'] + counts['FP'] + counts['TN'] + counts['FN']
    check = '✓' if total == counts['total'] else f'✗ (expected {counts["total"]})'
    print(f"{model:<10} {counts['TP']:<8} {counts['FP']:<8} {counts['TN']:<12} {counts['FN']:<8} {total} {check}")

# 검증: Precision = TP / (TP+FP) (GT 있는 쌍만 따로 검증)
if len(df_gt_pairs) > 0:
    print(f"\n[검증: Precision = TP / (TP+FP)] — GT 있는 쌍만")
    confusion_gt_only = {}
    for result in all_results:
        if not result['has_gt']:
            continue
        model = result['model']
        if model not in confusion_gt_only:
            confusion_gt_only[model] = {'TP': 0, 'FP': 0, 'TN': 0, 'FN': 0}
        confusion_gt_only[model]['TP'] += result['tp']
        confusion_gt_only[model]['FP'] += result['fp']
        confusion_gt_only[model]['TN'] += result['tn']
        confusion_gt_only[model]['FN'] += result['fn']
    
    for model in sorted(confusion_gt_only.keys()):
        counts = confusion_gt_only[model]
        tp_fp = counts['TP'] + counts['FP']
        if tp_fp > 0:
            precision_manual = counts['TP'] / tp_fp
            precision_lib = df_metrics[df_metrics['model'] == model]['Precision'].mean()
            print(f"  {model}: 계산={precision_manual:.4f}, 라이브러리={precision_lib:.4f}")

print(f"{'='*70}")

In [ ]:
# [cell-18] 강화된 평가 지표 (Precision@K, Recall@K, AP, MRR) - Valentine 버전
import numpy as np
from collections import defaultdict
from sklearn.metrics import roc_auc_score

K = 3
THRESHOLD = 0.5
k_values = [1, 3, 5, 10]

# ── GT 로드 ───────────────────────────────────────────────────────────────
gt_set = set()
for item in gt_all:
    ta, ca = item[0].split(".")
    tb, cb = item[1].split(".")
    gt_set.add(frozenset({(ta, ca), (tb, cb)}))

# ── 모델별 루프 ───────────────────────────────────────────────────────────
for model_name in sorted(set(r['model'] for r in all_results)):
    model_results = [r for r in all_results if r['model'] == model_name]

    # matches 객체 → score_map + 양방향 grouped
    grouped = defaultdict(list)
    score_map = {}

    for r in model_results:
        for (ta, ca), (tb, cb) in r['matches'].keys():
            score = r['matches'][((ta, ca), (tb, cb))]
            pair = frozenset({(ta, ca), (tb, cb)})
            score_map[pair] = score

            grouped[(ta, ca)].append({"A_table": ta, "A_col": ca,
                                      "B_table": tb, "B_col": cb, "FK_Score": score})
            grouped[(tb, cb)].append({"A_table": tb, "A_col": cb,
                                      "B_table": ta, "B_col": ca, "FK_Score": score})

    # ── Local Top-K 추출 ──────────────────────────────────────────────────
    local_topk_pairs = set()
    for (src_table, src_col), candidates in grouped.items():
        for r in sorted(candidates, key=lambda x: x["FK_Score"], reverse=True)[:K]:
            if r["FK_Score"] >= THRESHOLD:
                local_topk_pairs.add(frozenset({(r["A_table"], r["A_col"]),
                                                (r["B_table"], r["B_col"])}))

    # ── 기본 성능 지표 ────────────────────────────────────────────────────
    tp = len(gt_set & local_topk_pairs)
    fp = len(local_topk_pairs) - tp
    fn = len(gt_set) - tp

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    # ROC-AUC
    try:
        y_true  = [1 if p in gt_set else 0 for p in score_map]
        y_score = [score_map[p] for p in score_map]
        roc_auc = roc_auc_score(y_true, y_score) if sum(y_true) > 0 else float("nan")
    except Exception:
        roc_auc = float("nan")

    # ── Global Top-K (Precision@K, Recall@K) ─────────────────────────────
    all_pairs_sorted = sorted(score_map.items(), key=lambda x: x[1], reverse=True)
    precision_at_k, recall_at_k = {}, {}
    for k in k_values:
        top_k_pairs = {p for p, _ in all_pairs_sorted[:k]}
        tp_k = len(gt_set & top_k_pairs)
        precision_at_k[k] = tp_k / k
        recall_at_k[k] = tp_k / len(gt_set) if gt_set else 0.0

    # ── Average Precision (AP) ────────────────────────────────────────────
    ap_score, prev_recall = 0.0, 0.0
    for k, (pair, _) in enumerate(all_pairs_sorted, 1):
        if pair in gt_set:
            cur_recall = k / len(gt_set)
            cur_prec   = sum(1 for p, _ in all_pairs_sorted[:k] if p in gt_set) / k
            ap_score  += cur_prec * (cur_recall - prev_recall)
            prev_recall = cur_recall

    # ── Local Top-K Precision@K, Recall@K ────────────────────────────────
    local_prec_k = {k: [] for k in k_values}
    local_rec_k  = {k: [] for k in k_values}
    for (src_table, src_col), candidates in grouped.items():
        sorted_cands = sorted(candidates, key=lambda x: x["FK_Score"], reverse=True)
        src_gt = {p for p in gt_set if (src_table, src_col) in p}
        if not src_gt:
            continue
        for k in k_values:
            pred = {frozenset({(r["A_table"], r["A_col"]), (r["B_table"], r["B_col"])})
                    for r in sorted_cands[:k]}
            tp_k = len(src_gt & pred)
            local_prec_k[k].append(tp_k / k)
            local_rec_k[k].append(tp_k / len(src_gt))

    avg_lp = {k: np.mean(v) if v else 0.0 for k, v in local_prec_k.items()}
    avg_lr = {k: np.mean(v) if v else 0.0 for k, v in local_rec_k.items()}

    # ── MRR ──────────────────────────────────────────────────────────────
    mrr_scores = []
    for col, cands in grouped.items():
        for rank, r in enumerate(sorted(cands, key=lambda x: x["FK_Score"], reverse=True), 1):
            if frozenset({(r["A_table"], r["A_col"]), (r["B_table"], r["B_col"])}) in gt_set:
                mrr_scores.append(1.0 / rank)
                break
    mrr = np.mean(mrr_scores) if mrr_scores else 0.0

    # ── 출력 ─────────────────────────────────────────────────────────────
    print()
    print(f"{'='*70}")
    print(f"[{model_name}]  GT FK: {len(gt_set)}개  |  후보 컬럼: {len(grouped)}개")
    print(f"{'='*70}")

    print(f"\nLocal Top-{K} 성능 지표")
    print(f"  예측 FK 쌍: {len(local_topk_pairs)}개  |  TP={tp}, FP={fp}, FN={fn}")
    print(f"  Precision : {precision:.4f}  |  Recall : {recall:.4f}  |  F1 : {f1:.4f}")
    print(f"  ROC-AUC   : {roc_auc:.4f}")

    print(f"\nGlobal Top-K 성능 지표 (전체 쌍 기준)")
    for k in k_values:
        print(f"  Precision@{k:<2}: {precision_at_k[k]:.4f}  |  Recall@{k:<2}: {recall_at_k[k]:.4f}")

    print(f"\nLocal Top-K 성능 지표 (각 컬럼별 평균)")
    for k in k_values:
        print(f"  Precision@{k:<2}: {avg_lp[k]:.4f}  |  Recall@{k:<2}: {avg_lr[k]:.4f}")

    print(f"\n추가 평가 지표")
    print(f"  Average Precision (AP) : {ap_score:.4f}  |  MRR : {mrr:.4f}")

    print(f"\nGT FK 쌍 탐지 여부 (Local Top-{K})")
    for pair in sorted(gt_set, key=str):
        t0, c0 = sorted(pair)[0]; t1, c1 = sorted(pair)[1]
        print(f"  {'✅' if pair in local_topk_pairs else '❌'}  {t0}.{c0:20} ↔ {t1}.{c1:20}")

    topk_sorted = sorted(local_topk_pairs, key=lambda p: score_map.get(p, 0), reverse=True)
    print(f"\n예측된 FK 쌍 (총 {len(topk_sorted)}개)")
    print(f"  {'Score':>7}  {'A':35}  {'B':35}  GT")
    for pair in topk_sorted[:20]:
        t0, c0 = sorted(pair)[0]; t1, c1 = sorted(pair)[1]
        a_str = f"{t0}.{c0}"[:35]
        b_str = f"{t1}.{c1}"[:35]
        gt_flag = "✅" if pair in gt_set else "❌"
        print(f"  {score_map.get(pair, 0):>7.4f}  {a_str:35}  {b_str:35}  {gt_flag}")
    if len(topk_sorted) > 20:
        print(f"  ... 외 {len(topk_sorted)-20}개")


In [ ]:
# ── CRIM 평가 프로토콜 호환 ────────────────────────────────────
# CRIM_0517_tpc_only_train_v34_final.ipynb과 동일한 방식으로 P/R/F1 계산
#   1) 명시적 임계값 (THRESHOLD) cutoff — 매처 내부 임계값 무시
#   2) frozenset 기반 순서 무관 매칭
#   3) 폴더 전체 컬럼쌍에 대해 micro 집계 (쌍별 평균 X)
#   4) tn = pair_count - tp - fp - fn (Valentine cell-13의 1:1 필터 미적용)

THRESHOLD = 0.5

# ── GT를 frozenset 형태로 변환 (CRIM의 gt_canonical과 동일 구조) ─────
gt_canonical = set()
for item in gt_all:
    ta, ca = item[0].split('.')
    tb, cb = item[1].split('.')
    gt_canonical.add(frozenset({(ta, ca), (tb, cb)}))

print(f"\n{'='*70}")
print(f"[CRIM 호환 평가] threshold={THRESHOLD}")
print(f"  DB: {db_name}")
print(f"  GT FK 쌍: {len(gt_canonical)}개")
print(f"{'='*70}")

# ── 모델별 집계 ──────────────────────────────────────────────────
crim_style_summary = {}

for model_name in sorted(set(r['model'] for r in all_results)):
    model_results = [r for r in all_results if r['model'] == model_name]

    # 전체 컬럼쌍 수 = 폴더 내 모든 CSV 쌍의 (cols_a × cols_b) 합산
    pair_count = sum(r['total_possible'] for r in model_results)

    # 예측: score >= THRESHOLD 만 채택 (CRIM과 동일)
    pred_canonical = set()
    for r in model_results:
        for ((ta, ca), (tb, cb)), score in r['matches'].items():
            if score >= THRESHOLD:
                pred_canonical.add(frozenset({(ta, ca), (tb, cb)}))

    tp = len(gt_canonical & pred_canonical)
    fp = len(pred_canonical) - tp
    fn = len(gt_canonical) - tp
    tn = pair_count - tp - fp - fn

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    crim_style_summary[model_name] = {
        'pair_count': pair_count,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'precision': precision, 'recall': recall, 'f1': f1,
    }

    print(f"\n[{model_name}]")
    print(f"  전체 컬럼 쌍 : {pair_count}")
    print(f"  TP={tp}, FP={fp}, FN={fn}, TN={tn}")
    print(f"  Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

# ── 요약 테이블 (CRIM 출력 포맷과 유사) ───────────────────────────
print(f"\n{'='*70}")
print(f"[CRIM 호환 요약 테이블]  threshold={THRESHOLD}")
print(f"{'='*70}")
print(f"{'Model':<8} {'TP':>4} {'FP':>5} {'FN':>4} {'TN':>6}  "
      f"{'Precision':>10} {'Recall':>8} {'F1':>8}")
print(f"{'-'*70}")
for model_name in sorted(crim_style_summary.keys()):
    s = crim_style_summary[model_name]
    print(f"{model_name:<8} {s['tp']:>4} {s['fp']:>5} {s['fn']:>4} {s['tn']:>6}  "
          f"{s['precision']:>10.4f} {s['recall']:>8.4f} {s['f1']:>8.4f}")

# ── CRIM 결과 형식으로 저장 (직접 비교용) ────────────────────────
crim_compat_output = {
    'db': db_name,
    'threshold': THRESHOLD,
    'n_gt': len(gt_canonical),
    'per_model': {
        m: {
            'precision': round(s['precision'], 4),
            'recall':    round(s['recall'], 4),
            'f1':        round(s['f1'], 4),
            'tp': s['tp'], 'fp': s['fp'], 'fn': s['fn'], 'tn': s['tn'],
        }
        for m, s in crim_style_summary.items()
    }
}

crim_output_path = f'../workspace/valentine_results_crim_compat_{db_name}.json'
with open(crim_output_path, 'w') as f:
    json.dump(crim_compat_output, f, indent=2)
print(f"\n[CRIM 호환 결과 저장] {crim_output_path}")


In [ ]:
# ── PR-AUC + Recall@K (임계값-독립 공정 비교 지표) ───────────────
# CRIM / Valentine / EmbDI / DeepJoin 같은 이질적 모델을 한 표에서 비교
# 하려면 임계값에 의존하지 않는 ranking 지표가 필요함.
#   1) PR-AUC : 극도로 불균형한 FK 탐지에 적합 (Valentine SIGMOD 2021)
#   2) Recall@K (K=|GT|) : Valentine 논문 메인 지표
#     ※ K=|GT| 일 때 Precision@K = Recall@K = F1@K (동일 분모)
#
# 미스코어 쌍 (matcher 내부 임계값으로 누락된 쌍) 은 score=0 가정
# CRIM과 비교할 때도 같은 방식 — CRIM은 모든 쌍에 score 출력하므로 호환됨

from sklearn.metrics import average_precision_score
import numpy as np

# ── GT를 frozenset 형태로 변환 ─────────────────────────────────
gt_canonical = set()
for item in gt_all:
    ta, ca = item[0].split('.')
    tb, cb = item[1].split('.')
    gt_canonical.add(frozenset({(ta, ca), (tb, cb)}))

K = len(gt_canonical)

print(f"\n{'='*70}")
print(f"[임계값-독립 평가 지표] DB: {db_name}")
print(f"  GT FK 쌍: {K}개  |  Recall@K (K={K})")
print(f"{'='*70}")

# ── 모든 컬럼쌍 후보 enumerate (CSV 재로드) ────────────────────
all_candidate_pairs = []  # list of frozenset
for path_a, path_b in csv_pairs:
    name_a, name_b = path_a.stem, path_b.stem
    try:
        df_a = clean_df_for_valentine(pd.read_csv(path_a))
        df_b = clean_df_for_valentine(pd.read_csv(path_b))
    except Exception as e:
        print(f"[경고] {name_a} × {name_b} 로드 실패: {e}")
        continue
    for col_a in df_a.columns:
        for col_b in df_b.columns:
            all_candidate_pairs.append(
                frozenset({(name_a, col_a), (name_b, col_b)})
            )

total_pairs = len(all_candidate_pairs)
n_pos = sum(1 for p in all_candidate_pairs if p in gt_canonical)
print(f"  전체 후보 쌍 : {total_pairs}")
print(f"  양성 쌍 (GT)  : {n_pos}  (양성률 {n_pos/total_pairs*100:.3f}%)")

# ── 모델별 평가 ──────────────────────────────────────────────
ranking_summary = {}

for model_name in sorted(set(r['model'] for r in all_results)):
    model_results = [r for r in all_results if r['model'] == model_name]

    # 점수 맵 (frozenset -> score). 같은 쌍이 여러 csv_pair에 안 걸쳐 있어
    # 충돌은 없지만 안전하게 max 사용.
    score_map = {}
    for r in model_results:
        for ((ta, ca), (tb, cb)), score in r['matches'].items():
            pair = frozenset({(ta, ca), (tb, cb)})
            score_map[pair] = max(score_map.get(pair, 0.0), score)

    # y_true / y_score (전체 후보 순서 고정)
    y_true  = np.array([1 if p in gt_canonical else 0 for p in all_candidate_pairs])
    y_score = np.array([score_map.get(p, 0.0) for p in all_candidate_pairs])

    # PR-AUC
    pr_auc = average_precision_score(y_true, y_score) if y_true.sum() > 0 else float('nan')

    # Recall@K (K = |GT|)
    if K > 0 and K <= total_pairs:
        top_k_idx = np.argsort(-y_score, kind='stable')[:K]
        tp_at_K = int(y_true[top_k_idx].sum())
        recall_at_K = tp_at_K / K
    else:
        tp_at_K, recall_at_K = 0, float('nan')

    # Recall@3K (extra — 후보 폭 조금 더 넓게)
    if K > 0 and 3*K <= total_pairs:
        top_3k_idx = np.argsort(-y_score, kind='stable')[:3*K]
        recall_at_3K = int(y_true[top_3k_idx].sum()) / K
    else:
        recall_at_3K = float('nan')

    ranking_summary[model_name] = {
        'pr_auc':       pr_auc,
        'recall_at_K':  recall_at_K,
        'recall_at_3K': recall_at_3K,
        'tp_at_K':      tp_at_K,
        'n_scored':     int((y_score > 0).sum()),
        # PR curve 그리기용 raw 데이터
        'y_true':       y_true.astype(int).tolist(),
        'y_score':      [round(float(s), 6) for s in y_score],
    }

    print(f"\n[{model_name}]")
    print(f"  점수가 있는 쌍 : {int((y_score > 0).sum()):>4} / {total_pairs}")
    print(f"  PR-AUC         : {pr_auc:.4f}")
    print(f"  Recall@{K:<3}    : {recall_at_K:.4f}  ({tp_at_K}/{K})")
    if not np.isnan(recall_at_3K):
        print(f"  Recall@{3*K:<3}    : {recall_at_3K:.4f}")

# ── 요약 테이블 ──────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"[공정 비교 요약 — 임계값-독립]")
print(f"{'='*70}")
print(f"{'Model':<10} {'PR-AUC':>8} {'R@'+str(K):>8} {'R@'+str(3*K):>8} "
      f"{'#scored':>10}")
print(f"{'-'*70}")
for model_name in sorted(ranking_summary.keys()):
    s = ranking_summary[model_name]
    r3k_str = f"{s['recall_at_3K']:.4f}" if not np.isnan(s['recall_at_3K']) else "   N/A"
    print(f"{model_name:<10} {s['pr_auc']:>8.4f} {s['recall_at_K']:>8.4f} "
          f"{r3k_str:>8} {s['n_scored']:>10}")

# ── 저장 (CRIM/EmbDI/DeepJoin 결과와 옆에 두고 비교용) ──────────
ranking_output_path = (
    f'../workspace/'
    f'valentine_ranking_metrics_{db_name}.json'
)
with open(ranking_output_path, 'w') as f:
    json.dump({
        'db':          db_name,
        'n_gt':        K,
        'total_pairs': total_pairs,
        'per_model': {
            m: {
                'pr_auc':       round(s['pr_auc'], 4),
                'recall_at_K':  round(s['recall_at_K'], 4),
                'recall_at_3K': (round(s['recall_at_3K'], 4)
                                 if not np.isnan(s['recall_at_3K']) else None),
                'tp_at_K':      s['tp_at_K'],
                'n_scored':     s['n_scored'],
                'y_true':       s.get('y_true'),
                'y_score':      s.get('y_score'),
            } for m, s in ranking_summary.items()
        }
    }, f, indent=2)
print(f"\n[저장] {ranking_output_path}")


In [ ]:
# ── Classic metrics (TP/FP/FN/TN, Precision/Recall/F1) 저장 ─────────
# CRIM 호환 셀에서 계산된 crim_style_summary 를 _classic 파일로 별도 저장.
# 통합 노트북 또는 표 작성 시 ranking 지표(PR-AUC/R@K) 와 함께 보기 위함.

import json

classic_path = (
    f'../workspace/'
    f'valentine_ranking_metrics_{db_name}_classic.json'
)

classic_data = {
    'db':        db_name,
    'threshold': THRESHOLD,
    'n_gt':      len(gt_canonical),
    'per_model': {
        model: {
            'tp':          int(s['tp']),
            'fp':          int(s['fp']),
            'fn':          int(s['fn']),
            'tn':          int(s['tn']),
            'precision':   round(s['precision'], 4),
            'recall':      round(s['recall'], 4),
            'f1':          round(s['f1'], 4),
            'total_pairs': int(s['pair_count']),
        } for model, s in crim_style_summary.items()
    }
}

with open(classic_path, 'w') as f:
    json.dump(classic_data, f, indent=2)
print(f"[저장] {classic_path}")